### CEO Genie Spaces

Creates two Databricks Genie spaces for CEO-level analytics:
- **Revenue & Orders Genie** — real-time revenue, order volume, and location performance from Lakeflow gold tables
- **Operations Intelligence Genie** — kitchen ops, event patterns, location health from Lakeflow pipeline tables

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

In [ ]:
from databricks.sdk import WorkspaceClient
import json, hashlib, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

# Create or reuse a shared SQL warehouse for Genie
WAREHOUSE_NAME = f"{CATALOG}-ceo-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"\u267b\ufe0f Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"\u2705 Created warehouse: {warehouse.id}")

##### Helper: create or reuse a Genie space

In [ ]:
def make_questions(questions):
    return [
        {"id": hashlib.md5(q.encode()).hexdigest(), "question": [q]}
        for q in questions
    ]


def create_or_get_genie(title, description, tables, questions, state_key=None):
    """Create a Genie space, reusing existing if found in uc_state."""
    genie_space_id = None

    if state_key:
        try:
            df = spark.sql(f"""
                SELECT resource_data FROM {CATALOG}._internal_state.resources
                WHERE resource_type = 'genie_spaces'
                ORDER BY created_at DESC
            """)
            for row in df.collect():
                info = json.loads(row.resource_data)
                if info.get("title") == title:
                    candidate_id = info.get("space_id")
                    if candidate_id:
                        try:
                            w.genie.get_space(candidate_id)
                            genie_space_id = candidate_id
                            print(f"\u267b\ufe0f Reusing existing Genie space: {genie_space_id}")
                        except Exception:
                            pass
        except Exception:
            pass

    if not genie_space_id:
        serialized_space = json.dumps({
            "version": 2,
            "data_sources": {
                "tables": [{"identifier": t} for t in sorted(tables)],
            },
            "config": {"sample_questions": make_questions(questions)},
        })
        space = w.genie.create_space(
            warehouse_id=warehouse.id,
            serialized_space=serialized_space,
            title=title,
            description=description,
        )
        genie_space_id = space.space_id
        print(f"\u2705 Created Genie space: {genie_space_id}")

    add(CATALOG, "genie_spaces", {"space_id": genie_space_id, "title": title})
    return genie_space_id

##### Revenue & Orders Genie

In [ ]:
REVENUE_TITLE = f"CEO Revenue & Orders Intelligence ({CATALOG})"
REVENUE_DESC = f"""You are a revenue analytics assistant for the CEO of Casper's Kitchens.
Answer using pre-aggregated Gold tables first — they are MUCH faster than all_events.

=== PREFERRED TABLES (always try these first) ===

{CATALOG}.lakeflow.gold_location_sales_hourly:
  Hourly revenue per location. Columns: location_id (string), hour_ts (timestamp), orders (bigint), revenue (double).
  Use for: total revenue, orders by location, time-series revenue, recent period comparisons.
  Example: SELECT l.name, SUM(revenue) FROM gold_location_sales_hourly g JOIN simulator.locations l ON g.location_id=l.location_id WHERE hour_ts >= current_timestamp()-INTERVAL 7 DAYS GROUP BY l.name

{CATALOG}.lakeflow.gold_brand_sales_day:
  Daily revenue per brand. Columns: brand_id (int), day (date), orders (bigint), items_sold (bigint), brand_revenue (double).
  Use for: brand performance, top brands, day-over-day trends.

{CATALOG}.lakeflow.gold_order_header:
  One row per order with pre-computed revenue. Columns: order_id (string), location_id (string), order_day (date), order_revenue (double), total_qty (bigint), total_items (bigint).
  Use for: average order value, order count, per-order analysis.

{CATALOG}.simulator.locations:
  Location dimension. Columns: location_id (string), name, city, state, latitude, longitude.

{CATALOG}.simulator.brands:
  Brand dimension. Columns: brand_id (int), name, cuisine.

=== FALLBACK (only if Gold tables cannot answer) ===

{CATALOG}.lakeflow.all_events:
  Raw event stream — 1M+ rows, requires JSON parsing. Avoid unless specifically needed.

=== QUERY GUIDELINES ===
- For revenue by location → use gold_location_sales_hourly JOIN simulator.locations
- For brand performance → use gold_brand_sales_day JOIN simulator.brands
- For average order value → use gold_order_header
- Always filter by date range when possible (reduces scan cost dramatically)
- Revenue = SUM(revenue) from gold tables (already pre-computed)
- For last 7 days: WHERE hour_ts >= current_timestamp() - INTERVAL 7 DAYS"""

REVENUE_TABLES = [
    # Gold tables first - pre-aggregated, 10-100x faster for revenue queries
    f"{CATALOG}.lakeflow.gold_location_sales_hourly",
    f"{CATALOG}.lakeflow.gold_brand_sales_day",
    f"{CATALOG}.lakeflow.gold_order_header",
    f"{CATALOG}.lakeflow.silver_order_items",
    # Raw + dimension tables as fallback
    f"{CATALOG}.lakeflow.all_events",
    f"{CATALOG}.simulator.locations",
    f"{CATALOG}.simulator.brands",
    f"{CATALOG}.simulator.items",
    f"{CATALOG}.simulator.brand_locations",
]

REVENUE_QUESTIONS = [
    "What is total revenue by location for the last 30 days?",
    "Which brand generates the most orders across all locations?",
    "What is the order cancellation rate by location?",
    "Show me hourly order volume trends for today",
    "Which location has the highest average order value?",
    "What is the revenue split between locations this week vs last week?",
    "How many total orders were placed across all locations?",
    "Which brands are underperforming relative to the network average?",
]

revenue_genie_id = create_or_get_genie(
    title=REVENUE_TITLE,
    description=REVENUE_DESC,
    tables=REVENUE_TABLES,
    questions=REVENUE_QUESTIONS,
    state_key="revenue",
)
print(f"Revenue Genie ID: {revenue_genie_id}")

##### Operations Intelligence Genie

In [ ]:
OPS_TITLE = f"CEO Operations Intelligence ({CATALOG})"
OPS_DESC = f"""You are an operations analytics assistant for the CEO of Casper's Kitchens.
Answer using the most efficient table for each question type.

=== PREFERRED TABLES ===

{CATALOG}.lakeflow.gold_location_sales_hourly:
  Hourly ops metrics per location. Columns: location_id (string), hour_ts (timestamp), orders (bigint), revenue (double).
  Use for: order volume trends, location throughput, peak hour analysis.

{CATALOG}.lakeflow.gold_order_header:
  One row per order. Columns: order_id (string), location_id (string), order_day (date), order_revenue (double), total_qty (bigint).
  Use for: order counts, per-order metrics.

{CATALOG}.lakeflow.silver_order_items:
  Order line items. Columns: order_id, location_id, order_ts (timestamp), order_day (date), item_id, brand_id, item_name (string), price (double).
  Use for: item-level operations, brand mix, menu performance.

{CATALOG}.food_safety.inspections:
  Inspection records. Columns: inspection_id, location_id, location_name, inspection_date, score, grade, violation_count, critical_count.
  Use for: food safety scores, inspection grades, compliance status.

{CATALOG}.food_safety.violations:
  Violation details. Columns: inspection_id, location_id, inspection_date, code, severity, category, description.
  Use for: specific violations, severity analysis.

{CATALOG}.simulator.locations:
  Location dimension. Columns: location_id (string), name, city, state.

=== FALLBACK ===

{CATALOG}.lakeflow.all_events:
  Raw 1M-row event stream. Use ONLY for event-type-specific analysis.
  event_type values: order_created, delivered, gk_started, gk_finished, gk_ready, driver_ping, driver_arrived, driver_picked_up.

=== CANCEL RATE COMPUTATION ===
Cancellation rate is not a pre-aggregated column. Compute it from all_events:
  cancelled = orders with order_created event but NO delivered event.
  Use this pattern:
    SELECT l.name,
      COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END) AS total_orders,
      COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
        - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END) AS cancelled_orders,
      ROUND(100.0 * (
        COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
        - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END)
      ) / NULLIF(COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END), 0), 1) AS cancel_rate_pct
    FROM {CATALOG}.lakeflow.all_events ae
    JOIN {CATALOG}.simulator.locations l ON ae.location_id = l.location_id
    WHERE ae.event_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name ORDER BY cancel_rate_pct DESC

=== QUERY GUIDELINES ===
- Prefer Gold/Silver tables over all_events (10-100x faster)
- Always filter by date range when possible
- For location names, JOIN simulator.locations on location_id
- For cancel rate, always use the all_events pattern above — do NOT return 0 without running this query"""

OPS_TABLES = [
    f"{CATALOG}.lakeflow.gold_location_sales_hourly",
    f"{CATALOG}.lakeflow.gold_order_header",
    f"{CATALOG}.lakeflow.silver_order_items",
    f"{CATALOG}.lakeflow.all_events",
    f"{CATALOG}.simulator.locations",
    f"{CATALOG}.food_safety.inspections",
    f"{CATALOG}.food_safety.violations",
]

OPS_QUESTIONS = [
    "Which locations have the highest complaint rate?",
    "What is the refund approval rate by location?",
    "Show me locations with critical food safety violations",
    "What are the top 5 complaint categories across all locations?",
    "Which location needs the most operational attention right now?",
    "Show order cancellation trends over the last 7 days by location",
    "What is the average food safety inspection score by location?",
    "How many complaints were submitted in the last 24 hours?",
]

ops_genie_id = create_or_get_genie(
    title=OPS_TITLE,
    description=OPS_DESC,
    tables=OPS_TABLES,
    questions=OPS_QUESTIONS,
    state_key="ops",
)
print(f"Operations Genie ID: {ops_genie_id}")

print(f"\n\u2705 CEO Genie Spaces stage complete")
print(f"   Revenue Genie: {revenue_genie_id}")
print(f"   Operations Genie: {ops_genie_id}")